# msd_lib — Litinski 15-to-1 蒸留のライブラリ（関数定義のみ）

`%run -i ./common_lib.ipynb` の後に `%run -i ./msd_lib.ipynb` で読み込む。

**一次資料**：Litinski, *A Game of Surface Codes* [Lit]：式(1)(2)(4)、Fig. 14, 15, 17, 18, 37–41、Sec. 1（tile↔物理対応）、Sec. 3.3（11 tiles × 11τ, $121d^3$）、App. A（状態注入）。

内容
1. 式(1)(2)(4)の行列（論文からの転記）と三直交性
2. 状態ベクトルによる 5量子ビット回路（Fig. 15）、15量子ビット符号（Fig. 14）、auto-corrected π/8 回転（Fig. 17b）
3. Fig. 17c の tile 配置（転記）
4. 回転表面符号パッチ・格子手術（stim による物理レベル検証用）
5. 15-to-1 の段階別コスト表（tile モデル／物理モデル）

In [ ]:
"""Litinski 'A Game of Surface Codes' Sec. 3: 15-to-1 distillation.
Logical-level circuits (statevector) + tile-level layout model + physical (patch-level) model."""
import itertools
import numpy as np

### matrices (transcribed)

In [ ]:
# ----------------------------------------------------------------- matrices (transcribed)
# Eq. (1) of Litinski: rows 1-4 = X stabilizers, row 5 = logical X, columns = 15 qubits
M15 = np.array([
    [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1],
    [0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1],
    [1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1],
    [0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1]], dtype=np.uint8)
# Eq. (2): triorthogonal matrix G ; Eq. (4): its row-echelon form G~
G_TRI = np.array([[1] * 16,
                  [0] * 8 + [1] * 8,
                  [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1],
                  [0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1],
                  [0, 1] * 8], dtype=np.uint8)
G_TILDE = np.array([
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1],
    [0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1],
    [0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1],
    [1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1]], dtype=np.uint8)


def triorthogonal(G):
    G = G.astype(int)
    a = all(G[i].sum() % 8 == 0 for i in range(len(G)))
    b = all((G[i] * G[j]).sum() % 4 == 0 for i, j in itertools.combinations(range(len(G)), 2))
    c = all((G[i] * G[j] * G[k]).sum() % 2 == 0 for i, j, k in itertools.combinations(range(len(G)), 3))
    return a, b, c

### statevector helpers

In [ ]:
# ----------------------------------------------------------------- statevector helpers
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.diag([1, -1]).astype(complex)
PAULI = {'I': I2, 'X': X, 'Y': Y, 'Z': Z}
KET0 = np.array([1, 0], dtype=complex)
KETP = np.array([1, 1], dtype=complex) / np.sqrt(2)
T_GATE = np.diag([1, np.exp(1j * np.pi / 4)])
KET_M = T_GATE @ KETP                 # |m> = |0> + e^{i pi/4}|1>  (normalised)


def kron_all(ops):
    out = np.array([[1]], dtype=complex)
    for o in ops:
        out = np.kron(out, o)
    return out


def pauli_on(n, paulistr_by_qubit):
    """Dense Pauli operator; paulistr_by_qubit: dict qubit -> 'X'|'Y'|'Z'.  Qubit 0 = most significant."""
    return kron_all([PAULI[paulistr_by_qubit.get(i, 'I')] for i in range(n)])


def apply_1q(psi, n, q, U):
    psi = psi.reshape([2] * n)
    psi = np.moveaxis(np.tensordot(U, psi, axes=([1], [q])), 0, q)
    return psi.reshape(-1)


def z_rotation_diag(n, qubits, phi):
    """Diagonal of exp(-i phi Z_{q1} Z_{q2} ...)."""
    idx = np.arange(2 ** n)
    par = np.zeros(2 ** n, dtype=int)
    for q in qubits:
        par ^= (idx >> (n - 1 - q)) & 1
    return np.exp(-1j * phi * (1 - 2 * par))


def measure_pauli(psi, P, rng=None, forced=None):
    """Projective measurement of Hermitian Pauli matrix P.  Returns (outcome +-1, post-state, prob)."""
    Ppsi = P @ psi
    p_plus = np.real(np.vdot(psi, psi + Ppsi)) / 2
    if forced is None:
        out = 1 if rng.random() < p_plus else -1
    else:
        out = forced
    post = (psi + out * Ppsi) / 2
    prob = p_plus if out == 1 else 1 - p_plus
    if prob > 1e-12:
        post = post / np.sqrt(prob)
    return out, post, prob


def equal_up_to_phase(a, b, tol=1e-9):
    ov = np.vdot(a, b)
    return abs(abs(ov) - np.linalg.norm(a) * np.linalg.norm(b)) < tol

### 15-to-1, 5-qubit circuit (Fig. 15)

In [ ]:
# ----------------------------------------------------------------- 15-to-1, 5-qubit circuit (Fig. 15)
def fig15_rotations():
    """The 11 Z-type pi/8 rotations of Fig. 15 = columns 5..15 of M15 (rows = qubits 1..5).
    Columns 1..4 (weight-1 on qubits 4,3,2,1) are absorbed into the |m> initial states."""
    rots = []
    for j in range(4, 15):
        rots.append(tuple(int(i) for i in np.nonzero(M15[:, j])[0]))
    return rots


def run_15to1_5qubit(inputs=None, phi=np.pi / 8, rng=None):
    """Simulate Fig. 15.  inputs: list of 4 single-qubit states for qubits 1-4 (default ideal |m>).
    Returns dict outcome-tuple -> (probability, output state of qubit 5)."""
    n = 5
    if inputs is None:
        inputs = [KET_M] * 4
    psi = kron_all([s.reshape(2, 1) for s in inputs] + [KETP.reshape(2, 1)]).reshape(-1)
    for qs in fig15_rotations():
        psi = z_rotation_diag(n, qs, phi) * psi
    # measure qubits 1-4 in X basis: enumerate all outcomes
    res = {}
    for outs in itertools.product((1, -1), repeat=4):
        st = psi.copy()
        prob = 1.0
        for q, o in enumerate(outs):
            _, st, p = measure_pauli(st, pauli_on(n, {q: 'X'}), forced=o)
            prob *= p
            if p < 1e-14:
                break
        if prob < 1e-14:
            res[outs] = (0.0, None)
            continue
        # reduced state of qubit 5 (qubits 1-4 are in X eigenstates -> product)
        st = st.reshape(16, 2)
        # project onto the X-eigenstates of qubits 1..4
        xs = [np.array([1, o], dtype=complex) / np.sqrt(2) for o in outs]
        bra = kron_all([x.reshape(1, 2) for x in xs]).reshape(-1)
        out5 = bra.conj() @ st
        out5 = out5 / np.linalg.norm(out5)
        res[outs] = (prob, out5)
    return res

### 15-qubit code check (Fig. 14)

In [ ]:
# ----------------------------------------------------------------- 15-qubit code check (Fig. 14)
def rm15_logical_check(gate='T'):
    """Prepare |+_L> of the 15-qubit code (X stabilisers = rows 1-4 of M15, logical X = row 5,
    Z stabilisers = F2-orthogonal complement), apply transversal T (or T^dag) and return the
    logical expectation values <X_L>, <Y_L-like> to identify the logical gate."""
    n = 15
    from_rows = M15.astype(int)
    # Z stabilisers: nullspace of M15 over F2
    Hz = f2_nullspace_simple(M15)
    dim = 2 ** n
    psi = np.zeros(dim, dtype=complex)
    psi[0] = 1.0
    idx = np.arange(dim)
    # |+_L> = prod (1+X_s)/2 (1+X_L)/2 |0..0>  (|0..0> is +1 eigenstate of all Z stabilisers)
    for row in from_rows:
        mask = int(''.join(str(b) for b in row), 2)
        psi = (psi + psi[idx ^ mask]) / 2
    psi /= np.linalg.norm(psi)
    # transversal gate
    w = np.array([bin(i).count('1') for i in idx])
    ph = np.exp(1j * np.pi / 4 * w) if gate == 'T' else np.exp(-1j * np.pi / 4 * w)
    psi = ph * psi
    # logical operators: X_L = X(row5), Z_L = Z on a vector z with z.row5 = 1, z.rows1-4 = 0
    xl = int(''.join(str(b) for b in from_rows[4]), 2)
    zvec = find_logical_z(M15)
    zl = int(''.join(str(b) for b in zvec), 2)
    par = np.array([bin(i & zl).count('1') % 2 for i in idx])
    Zl_psi = (1 - 2 * par) * psi
    Xl_psi = psi[idx ^ xl]
    # Y_L = i X_L Z_L
    Yl_psi = 1j * Zl_psi[idx ^ xl]
    ex = np.real(np.vdot(psi, Xl_psi))
    ey = np.real(np.vdot(psi, Yl_psi))
    ez = np.real(np.vdot(psi, Zl_psi))
    # stabiliser expectation values (X stabilisers should be lost only up to the Clifford
    # correction; Z stabilisers preserved)
    zst = []
    for hz in Hz:
        m = int(''.join(str(b) for b in hz), 2)
        par = np.array([bin(i & m).count('1') % 2 for i in idx])
        zst.append(np.real(np.vdot(psi, (1 - 2 * par) * psi)))
    return dict(X_L=ex, Y_L=ey, Z_L=ez, min_Zstab=min(zst), n_Zstab=len(Hz))


def f2_nullspace_simple(M):
    M = M.copy() % 2
    rows, cols = M.shape
    A = M.copy()
    piv = []
    r = 0
    for c in range(cols):
        p = next((i for i in range(r, rows) if A[i, c]), None)
        if p is None:
            continue
        A[[r, p]] = A[[p, r]]
        for i in range(rows):
            if i != r and A[i, c]:
                A[i] ^= A[r]
        piv.append(c)
        r += 1
    free = [c for c in range(cols) if c not in piv]
    basis = []
    for f in free:
        v = np.zeros(cols, dtype=np.uint8)
        v[f] = 1
        for i, p in enumerate(piv):
            if A[i, f]:
                v[p] = 1
        basis.append(v)
    return np.array(basis)


def find_logical_z(M):
    """z with M[0:4] z = 0, M[4] z = 1 (brute force over weight)."""
    for w in range(1, 16):
        for sup in itertools.combinations(range(15), w):
            z = np.zeros(15, dtype=np.uint8)
            z[list(sup)] = 1
            s = (M.astype(int) @ z) % 2
            if not s[:4].any() and s[4] == 1:
                return z
    raise RuntimeError

### auto-corrected pi/8 (Fig. 17b)

In [ ]:
# ----------------------------------------------------------------- auto-corrected pi/8 (Fig. 17b)
def auto_corrected_pi8(P_data, psi_data, rng=None):
    """Fig. 17b on k data qubits + |m> + |0>.  P_data: dict qubit->Pauli on the data.
    Enumerates all measurement branches; for each, returns the Pauli correction needed and
    whether the post-correction data state equals exp(-i pi/8 P)|psi>.
    Measurement sequence (Litinski Fig. 17b):
      1. measure P (x) Z_m  and  Z_m (x) Y_0   (commuting, simultaneous)
      2. if P(x)Z_m = +1: measure |0>-qubit in Z ; else in X     (selective pi/4, Fig. 17a)
      3. measure |m> in X
      4. Pauli correction P^{c} with c determined by the outcomes (P_{pi/2} = P up to phase)."""
    k = int(np.log2(len(psi_data)))
    n = k + 2
    m, a = k, k + 1
    psi = np.kron(np.kron(psi_data, KET_M), KET0)
    P1 = pauli_on(n, {**P_data, m: 'Z'})
    P2 = pauli_on(n, {m: 'Z', a: 'Y'})
    Pd = pauli_on(k, P_data)
    target = np.cos(np.pi / 8) * psi_data - 1j * np.sin(np.pi / 8) * (Pd @ psi_data)
    branches = []
    for o1 in (1, -1):
        for o2 in (1, -1):
            s1, st, p1 = measure_pauli(psi, P1, forced=o1)
            if p1 < 1e-14:
                continue
            s2, st, p2 = measure_pauli(st, P2, forced=o2)
            if p2 < 1e-14:
                continue
            basis = 'Z' if o1 == 1 else 'X'
            for o3 in (1, -1):
                s3, st3, p3 = measure_pauli(st, pauli_on(n, {a: basis}), forced=o3)
                if p3 < 1e-14:
                    continue
                for o4 in (1, -1):
                    s4, st4, p4 = measure_pauli(st3, pauli_on(n, {m: 'X'}), forced=o4)
                    if p4 < 1e-14:
                        continue
                    # extract data state: ancillas are in known product eigenstates
                    anc_m = np.array([1, o4], dtype=complex) / np.sqrt(2)
                    anc_a = (np.array([1, 0], dtype=complex) if o3 == 1 else np.array([0, 1], dtype=complex)) \
                        if basis == 'Z' else np.array([1, o3], dtype=complex) / np.sqrt(2)
                    bra = np.kron(anc_m, anc_a)
                    dat = st4.reshape(2 ** k, 4) @ bra.conj()
                    dat /= np.linalg.norm(dat)
                    ok0 = equal_up_to_phase(dat, target)
                    ok1 = equal_up_to_phase(Pd @ dat, target)
                    branches.append(dict(PZ=o1, ZY=o2, anc_basis=basis, anc=o3, Xm=o4,
                                         prob=p1 * p2 * p3 * p4,
                                         correction='I' if ok0 else ('P' if ok1 else 'FAIL')))
    return branches

### tile layout (Fig. 17c)

In [ ]:
# ----------------------------------------------------------------- tile layout (Fig. 17c)
# Grid of 3 rows x 5 columns (row 0 = top).  Each entry: label of the patch occupying the tile.
FIG17C_STEP1 = [
    [None, None, 'q1', 'q3', None],
    [None, None, None, None, 'q5'],
    [None, None, 'q2', 'q4', None]]
FIG17C_ROTATION = [
    ['zero', 'm', 'q1', 'q3', None],
    ['zero', 'm', 'anc', 'anc', 'q5'],
    [None, None, 'q2', 'q4', None]]
FIG17C_FINAL = [
    ['zero', 'm', 'q1', 'q3', None],
    ['zero', 'm', None, None, 'q5'],
    [None, None, 'q2', 'q4', None]]
# NOTE: in Step 1 the 2x2 block on the left is shaded as part of the block (reserved) and the
# |m> states for the rotations are prepared there; Litinski counts 11 tiles for the block.


def tile_schedule(n_rot=11):
    """List of (label, tau, layout) for the Fig. 17c protocol.  tau = time steps (1 = d cycles)."""
    sched = [('step1_init', 0, FIG17C_STEP1)]
    for r in range(n_rot):
        sched.append((f'rotation_{r + 1}', 1, FIG17C_ROTATION))
        sched.append((f'readout_{r + 1}', 0, FIG17C_FINAL))
    sched.append(('final_X_measurement', 0, FIG17C_FINAL))
    return sched


def tiles_used(sched):
    used = set()
    for _, _, lay in sched:
        for i, row in enumerate(lay):
            for j, lab in enumerate(row):
                if lab is not None:
                    used.add((i, j))
    return used


import numpy as np


def rotated_checks(rows, cols, origin=(0, 0)):
    """Stabilizers of a rotated planar code on data qubits (r,c), 0<=r<rows, 0<=c<cols (shifted by
    origin).  Check type from global parity (r+c)%2 of the plaquette corner -> consistent tiling.
    Weight-2 Z checks on top/bottom, weight-2 X checks on left/right, so that the logical Z
    string runs vertically, along the left/right boundaries (the boundaries used for Z(x)Z surgery)."""
    r0, c0 = origin
    checks = []
    for r in range(rows + 1):
        for c in range(cols + 1):
            qs = [(r0 + rr, c0 + cc) for rr in (r - 1, r) for cc in (c - 1, c)
                  if 0 <= rr < rows and 0 <= cc < cols]
            typ = 'X' if (r0 + r + c0 + c) % 2 == 0 else 'Z'
            if len(qs) == 4:
                checks.append((typ, tuple(qs)))
            elif len(qs) == 2:
                horizontal_edge = (r == 0 or r == rows)
                if (horizontal_edge and typ == 'Z') or ((c == 0 or c == cols) and typ == 'X'):
                    checks.append((typ, tuple(qs)))
    return checks


def patch_gadgets(checks, qindex, anc_start):
    gads = []
    a = anc_start
    for typ, qs in checks:
        idx = [qindex[q] for q in qs]
        if typ == 'X':
            gads.append(pauli_check_gadget(a, xq=idx))
        else:
            gads.append(z_check_gadget(a, idx))
        a += 1
    return gads


def patch_qubit_count(d):
    ch = rotated_checks(d, d)
    return d * d, len(ch)


def lattice_surgery_zz_check(d, gap=None, seed=0):
    """Two d x d rotated patches separated by `gap` columns (default d, i.e. one tile).
    Prepare both in |+_L>, perform one noiseless merge round on the full rectangle, and verify
    with stim that  Zbar1 Zbar2  becomes deterministic and equals the product of the merge-round
    outcomes of an F2 combination of merged Z checks.  Returns a dict of results."""
    import stim
    if gap is None:
        gap = d
    W = 2 * d + gap
    data = [(r, c) for r in range(d) for c in range(W)]
    qi = {q: i for i, q in enumerate(data)}
    p1 = rotated_checks(d, d, (0, 0))
    p2 = rotated_checks(d, d, (0, d + gap))
    merged = rotated_checks(d, W, (0, 0))
    nq = len(data)

    def vec(qs):
        v = np.zeros(nq, dtype=np.uint8)
        for q in qs:
            v[qi[q]] ^= 1
        return v

    def logical_z(checks, cols):
        # Z string along a column that commutes with the patch's X checks
        X = [vec(qs) for t, qs in checks if t == 'X']
        for c in cols:
            z = vec([(r, c) for r in range(d)])
            if all((x.astype(int) @ z) % 2 == 0 for x in X):
                return z
        raise RuntimeError('no column logical')

    z1 = logical_z(p1, range(0, d))
    z2 = logical_z(p2, range(d + gap, W))
    O = z1 ^ z2
    sim = stim.TableauSimulator()
    sim.set_num_qubits(nq)
    rng = np.random.default_rng(seed)
    # |+> on all data of both patches and on the gap (Litinski Fig. 41: ancilla region in |+>)
    for i in range(nq):
        sim.h(i)

    def meas(typ, qs):
        ps = stim.PauliString(nq)
        for q in qs:
            ps[qi[q]] = typ
        return sim.measure_observable(ps)

    pre = {}
    for t, qs in p1 + p2:
        pre[(t, qs)] = meas(t, qs)
    ps = stim.PauliString(nq)
    for i in np.nonzero(O)[0]:
        ps[int(i)] = 'Z'
    before = sim.peek_observable_expectation(ps)
    out = {}
    for t, qs in merged:
        out[(t, qs)] = meas(t, qs)
    after = sim.peek_observable_expectation(ps)
    Zm = [(k, vec(k[1])) for k in merged if k[0] == 'Z']
    A = np.array([v for _, v in Zm])
    coef = f2_solve(A, O)
    parity = 1
    for c, (k, _) in zip(coef, Zm):
        if c:
            parity *= -1 if out[k] else 1
    n_checks_merged = len(merged)
    return dict(d=d, gap=gap, data_qubits=nq, merged_checks=n_checks_merged,
                total_qubits=nq + n_checks_merged,
                exp_before=before, exp_after=after, product_of_outcomes=parity,
                consistent=(after == parity), O_in_span=coef is not None)

### Litinski (15-to-1)

In [ ]:
# ----------------------------------------------------------------- Litinski (15-to-1)
class MSDCostData:
    def __init__(self, d, dur=None):
        dur = dur or DUR_UNIT
        self.d = d
        ch = rotated_checks(d, d)
        qi = {(r, c): r * d + c for r in range(d) for c in range(d)}
        g = patch_gadgets(ch, qi, d * d)
        ub = round_schedule_classes(g, dur)
        lb = round_lower_bound(g, dur)
        self.round = dict(time_lb=lb['time_lb'], time_ub=ub['time_ub'])
        # exact patch counts
        self.sq_patch = d * d + len(ch)                                   # 2d^2 - 1
        ch2 = rotated_checks(d, 2 * d)
        self.tall_patch = 2 * d * d + len(ch2)                            # 4d^2 - 1
        self.cnot_sq = sum(len(qs) for _, qs in ch)
        self.cnot_tall = sum(len(qs) for _, qs in ch2)
        # ancilla region of the multi-patch measurement: 2 tiles; qubits added by merging,
        # approximated by the exact increment measured for a 1-tile gap (validated with stim):
        ch3 = rotated_checks(d, 3 * d)
        merged3 = 3 * d * d + len(ch3)
        self.gap_tile = merged3 - 2 * self.sq_patch                       # qubits added per gap tile
        self.cnot_gap_tile = sum(len(qs) for _, qs in ch3) - 2 * self.cnot_sq
        self.dur = dur


def msd_stages(md, model, params=DEFAULT_PARAMS, comparison='A', level='physical', data_only=False):
    """level='tile': Litinski's accounting (tiles x d^2 data qubits, or x 2d^2 incl. measurement
    qubits).  level='physical': exact rotated-patch qubit counts + measured gap-tile increment."""
    d = md.d
    tau = rounds_for(model, params['tau_rounds'], d)
    rows = []
    if level == 'tile':
        per_tile = d * d if data_only else 2 * d * d
        q_rot = 11 * per_tile
        q_idle = 9 * per_tile
        cnot_round = None
    else:
        if data_only:
            q_rot = 5 * d * d + 2 * (2 * d * d) + 2 * d * d   # patches + ancilla strip data (approx)
            q_idle = 5 * d * d + 2 * (2 * d * d)
        else:
            q_rot = 5 * md.sq_patch + 2 * md.tall_patch + 2 * md.gap_tile
            q_idle = 5 * md.sq_patch + 2 * md.tall_patch
        cnot_round = 5 * md.cnot_sq + 2 * md.cnot_tall + 2 * md.cnot_gap_tile
    meas_layer = md.dur['MEAS_X']

    def add(stage, qubits, rounds, extra_time, gates, source, note=''):
        rows.append(dict(protocol='Litinski_15to1', comparison=comparison, model=model, d=d, stage=stage,
                         qubits=qubits, rounds=rounds,
                         time_lb=rounds * md.round['time_lb'] + extra_time,
                         time_ub=rounds * md.round['time_ub'] + extra_time,
                         gates=dict(gates), source=source, note=note))

    inj = 0
    if comparison == 'B':
        inj = params['t_inj_FT'] if model == 'FT_ref' else params['t_inj_NL']
        add('L0 inject 4 |m> (qubits 1-4, parallel)', q_idle, inj, 0, Counter({'T': 4}), SOURCE_LIT,
            "Litinski App. A / Li: 3+3 cycles, d-independent; physical T gate per |m>")
    add('L1 init qubit 5 |+> (step 1)', q_idle, 0, 0, Counter(), SOURCE_LIT, '0 tau (Litinski)')
    for k in range(11):
        if comparison == 'B':
            add(f'L2.{k + 1}a inject |m> for rotation {k + 1}', q_idle, inj, 0, Counter({'T': 1}), SOURCE_LIT,
                'serial in the 11-tile layout (Litinski "Caveat")')
        g = Counter({'CNOT': cnot_round * tau}) if cnot_round else Counter()
        add(f'L2.{k + 1} auto-corrected pi/8 rotation {k + 1}', q_rot, tau, 0, g,
            SOURCE_LIT if level == 'tile' else SOURCE_COUNT,
            '1 tau = d cycles: P(x)Z_m and Z_m(x)Y_0 lattice surgery')
        add(f'L2.{k + 1}b readout |0>,|m> (single-qubit meas.)', q_idle, 0, meas_layer, Counter(),
            SOURCE_LIT, '0 tau; classical processing time not counted')
    add('L3 X measurement of qubits 1-4', q_idle, 0, meas_layer, Counter(), SOURCE_LIT, '0 tau')
    add('L4 logical X frame: T^dag|+> -> T|+>', 0, 0, 0, Counter(), SOURCE_COUNT, 'Pauli frame')
    return rows